# LangSmith

LangSmith는 LLM 애플리케이션이 한 요청을 처리한 과정을 기록해 원인 분석과 품질 개선에 쓰는 플랫폼이다.

모델 답변만 보면 결과가 왜 나왔는지 알기 어렵지만, LangSmith에서는 prompt 구성, 모델 호출, parser 처리, 도구 호출을 단계별로 따라갈 수 있다.

LangSmith는 OpenAI 전용이나 LangChain 전용 도구가 아니다. LangChain을 사용하면 지원 통합이 실행 정보를 자동으로 추적하며, 다른 프레임워크나 일반 Python 함수는 LangSmith SDK의 `@traceable`, `trace`, `RunTree`로 직접 계측할 수 있다. 이 수업에서는 LangChain의 자동 추적으로 시작해 trace를 읽고 비교하는 방법을 확인한다.


## Project, trace, run과 trace tree

**run**은 모델 호출, prompt 형식화, 문서 검색처럼 한 번 실행되는 작업 단위이다. 한 사용자의 요청을 처리하며 만들어진 run들의 묶음이 **trace**이며, trace는 부모·자식 run의 트리로 볼 수 있다. 같은 애플리케이션 또는 서비스의 trace를 모으는 상자가 **Project**이다.

아래 이미지는 application trace 아래에 도구 run과 `ChatOpenAI` model run이 중첩된 모습을 보여 준다.

- 최상위 trace에서는 전체 요청의 입력·출력·총 latency·error를 확인한다.
- child run에서는 각 단계의 입력·출력과 처리 시간을 확인한다.
- 뒤의 `Prompt → Model → Parser` 코드에서는 prompt·model·parser가 각각 child run으로 기록된다.

<img src="https://mintcdn.com/langchain-5e9cc07a/EKYgNtnIIDPnseTv/langsmith/images/trace-quickstart-app.png?fit=max&auto=format&n=EKYgNtnIIDPnseTv&q=85&s=35b19b5e1e978d13cdbb61058334eeb4" alt="LangSmith trace tree with child runs" width="1000"/>

`질문 → prompt → model → parser` 흐름에서는 전체 chain이 부모 run이다. prompt·model·parser는 자식 run이 된다.

- 공식 정의와 화면 용어: [Observability concepts](https://docs.langchain.com/langsmith/observability-concepts)
- 이미지의 실행 예제: [Observability quickstart](https://docs.langchain.com/langsmith/observability-quickstart)

## Observability와 evaluation은 서로 다른 질문에 답한다

**Observability**는 실제 한 실행에서 `무슨 일이 일어났는가`를 조사한다. 입력·출력, 지연 시간, 오류, 자식 run을 trace로 열어 느린 단계나 잘못 변환된 입력을 찾는 데 적합하다.

**Evaluation**은 `어느 버전이 더 좋은가`를 정한 기준으로 반복 측정한다.

- offline evaluation은 배포 전에 dataset으로 prompt나 모델 버전을 비교한다.
- online evaluation은 배포 후 실제 run·thread의 품질 신호를 확인한다.
- trace 두 개의 육안 비교를 dataset과 evaluator로 반복하면 정식 evaluation으로 확장할 수 있다.

평가 대상과 offline·online 경계는 [Evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts)에서 확인한다.


## LangSmith API Key, Project, `.env` 설정

- `OPENAI_API_KEY`: `ChatOpenAI`가 OpenAI 모델을 호출할 때 사용하는 인증 정보이다.
- `LANGSMITH_API_KEY`: trace를 LangSmith에 전송할 때 사용하는 인증 정보이다.
- `LANGSMITH_TRACING=true`: LangChain 실행의 자동 trace 전송을 켠다.
- `LANGSMITH_PROJECT`: trace를 모을 Project 이름이다. 생략하면 기본 Project에 기록된다.
- `LANGSMITH_ENDPOINT`: 기본 미국 리전이 아닌 계정에서 설정한다.
- `LANGSMITH_WORKSPACE_ID`: 한 API Key가 여러 workspace와 연결된 경우에만 설정한다.

`.env`는 프로젝트 최상위에 두고 Git에 올리지 않는다.

- API Key 생성과 리전 endpoint: [Create an account and API key](https://docs.langchain.com/langsmith/create-account-api-key)
- LangChain 자동 추적: [Trace LangChain applications](https://docs.langchain.com/langsmith/trace-with-langchain)

```text
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=<langsmith-api-key>
LANGSMITH_PROJECT=skn33-langsmith
# 기본 미국 리전이면 LANGSMITH_ENDPOINT를 생략한다.
# LANGSMITH_ENDPOINT=<account-specific-endpoint>
OPENAI_API_KEY=<openai-api-key>
OPENAI_CHAT_MODEL=<your-chat-model>
```

trace에는 입력과 출력이 저장될 수 있다. 개인정보·비밀키·고객 원문 대신 수업용 데이터를 사용하고, 필요한 경우 [입출력 숨김 설정](https://docs.langchain.com/langsmith/mask-inputs-outputs)을 적용한다.


### 필요한 패키지 설치

LangChain 모델 호출을 LangSmith에 자동으로 기록하려면 LangChain·OpenAI 통합과 LangSmith SDK가 필요하다. `%pip`는 현재 노트북 커널에 패키지를 설치하므로, PyCharm에서 선택한 인터프리터와 커널이 같은 환경인지 확인한다.


In [1]:
# %pip install -U langchain langchain_openai langsmith python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 인증과 추적 설정 불러오기

`.env`의 값을 현재 Python 프로세스 환경 변수로 등록한다.

In [5]:
import os
from dotenv import find_dotenv, load_dotenv
from nbclient.exceptions import stream_output_msg

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위에 .env 파일을 만든다.")
load_dotenv(dotenv_path, override=False)

required_names = ["OPENAI_API_KEY", "LANGSMITH_API_KEY", "OPENAI_CHAT_MODEL"]
missing_names = [name for name in required_names if not os.getenv(name)]
if os.getenv("LANGSMITH_TRACING", "").casefold() != "true":
    missing_names.append("LANGSMITH_TRACING=true")
if missing_names:
    raise RuntimeError(f".env에 다음 설정을 추가한다: {', '.join(missing_names)}")

# for name in required_names:
#     print(f"{name}\n={os.getenv(name)}")

OPENAI_CHAT_MODEL = os.environ["OPENAI_CHAT_MODEL"]
print("모델 호출과 LangSmith 추적 설정을 확인했다.")


모델 호출과 LangSmith 추적 설정을 확인했다.


## 단순 trace 호출

`ChatOpenAI.invoke()`에 문자열을 전달하면 LangChain이 채팅 모델을 호출하고 `AIMessage`를 반환한다. Responses API를 사용하면 `AIMessage.content`에는 text뿐 아니라 reasoning 같은 여러 content block이 들어갈 수 있다. 화면에 최종 답변만 출력할 때는 text block만 문자열로 꺼내는 `AIMessage.text`를 사용한다.

`LANGSMITH_TRACING=true`가 설정되어 있으면 별도 callback 코드를 추가하지 않아도 이 호출이 LangSmith에 trace로 기록된다. message 구조는 [LangChain Messages](https://docs.langchain.com/oss/python/langchain/messages)에서 확인한다.

In [12]:
from langchain_openai import ChatOpenAI

# ChatOpenAI() 객체 생성 시 환경변수 "OPEN_API_KEY"를 찾아와 사용
model = ChatOpenAI(  # OpenAI Chat 모델 전용 클래스
    # 모델 지정
    model=OPENAI_CHAT_MODEL,
    # 최신 responses api 사용 여부
    use_responses_api=True,
    # temperature=2.0
)

# question = "랭스미스의 trace가 무엇인지 두 문장으로 설명해줘"
question = "비 오는 날 서울을 배경으로 한 짧은 SF 이야기의 첫 문단을 써줘."

# 모델에게 question을 전달하고 응답(AI Message) 받는 구문
single_response = model.invoke(
    question,
    # 실행 설정 및 LangSmith 트래킹 데이터 지정
    config={
        # LangSmith 화면에 출력될 로그 이름
        "run_name": "single_model_call",
        # 검색, 필터링용 태그 지정
        "tegs": ["lecture", "single_call"],
        # 실행에 대한 추가적인 정보
        "metadata": {
            "input_style": "string",
            "lesson": "langchain_overview",
        },
    }
)

print(single_response.text)


비가 서울의 네온 간판들을 번지게 만들던 밤, 한강 위를 지나던 무인 택시들이 동시에 멈춰 섰다. 사람들은 단순한 통신 장애라고 생각했지만, 빗물에 젖은 모든 휴대전화 화면에는 같은 문장이 떠올랐다. **“서울 시민 여러분, 오늘 자정부터 도시는 여러분을 보호하기 위해 폐쇄됩니다.”**


### 기본 Project가 아닌 다른 Project에 trace 기록하기

`.env`의 `LANGSMITH_PROJECT`는 일반적인 LangChain 호출이 기록될 기본 Project를 지정한다. 특정 호출만 다른 Project에 분리하려면 `tracing_context()`의 `project_name`을 사용한다.

`tracing_context()`는 `.env`의 설정을 변경하지 않는다. `with` 블록 안의 호출만 `project_name`으로 지정한 Project에 기록하고, 블록 밖의 호출은 다시 기본 Project에 기록한다. 기능별 실행, prompt 버전 또는 테스트 환경의 trace를 분리할 때 사용할 수 있다.

```text
일반적인 model.invoke()
→ LANGSMITH_PROJECT에 기록

tracing_context() 내부의 model.invoke()
→ project_name으로 지정한 Project에 기록
```


In [18]:
import langsmith as ls

# .env에 지정된 LengSmith 기본 프로젝트가 아닌 다른 프로젝트명으로 모델 변경
# AI 모델을 변경할때 용이할것같음
other_project = "my_test"

# ls.tracing_context(): LengSmith 컨텍스트 매니저
with ls.tracing_context(project_name=other_project):
    other_response = model.invoke(
        "LangSmith에서 trace를 Project별로 분리하는 이유를 한 문장으로 설명해줘",
        config={"run_name": "other_model_call"},
    )

# open_model = ls.tracing_context(other_project)
# open_model.close()

print(other_response.text)

LangSmith에서 trace를 Project별로 분리하면 애플리케이션이나 실험 단위로 실행 기록을 체계적으로 관리하고 비교·분석할 수 있습니다.


### 역할이 있는 message 목록 추적하기

문자열 하나뿐 아니라 System·Human 역할을 분리한 message 목록도 같은 방식으로 추적할 수 있다. trace의 입력을 열면 역할별 message가 유지됐는지 확인할 수 있다.


In [15]:
translation_messages = [
    ("system", "주어진 한국어 문장을 자연스러운 영어 문장으로 번역한다, 비속어는 절절한 단어로 수정한다"),
    ("human", "이게 사람 날씨인가"),
]

translation_response = model.invoke(
    translation_messages,
    config={
        "run_name": "message_translation",
        "tegs": ["lecture", "messages"],
    },
)

print(translation_response.text)

Damn, it’s so hot.


## Prompt → Model → Parser 추적하기
* `|` : 버티컬바, 파이프는 LCEL이란 용어로 불린다
    * LangChain Expression Language : 파이프 연상자를 이용해서 prompt, model, parser 등의 변경을 한다


Runnable을 `|`(버티컬바, 파이프는 LCEL이란 용어로 불린다)로 연결하면 전체 chain이 부모 run이 되고 Prompt, Model, Parser가 자식 run으로 기록된다. 토큰 사용량은 모델 공급자를 실제로 호출한 model run에서 확인하며, parent chain의 합계가 표시되더라도 prompt·parser run의 사용량으로 오해하지 않는다. `with_config()`의 이름·태그·메타데이터는 여러 실험을 구분하는 검색 기준이 된다.


In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatPromptTemplate: AI(Chat Model)에 전달할 프롬프트를 생성하는 도구
# - 매번 문장을 새로 작성하지 않고 {}를 이용해 동적으로 내용을 채울 수 있음
# - System, Human, AI의 메시지 구조를 깔끔하게 다룰 수 있게함
# StrOutputParser: 복잡한 응답객체에서 응답 텍스트만 추출해주는 도구

# {question}에 내용을 전달 받은 내용을 집어넣고
# 각각을 System Message, Human Message로 변환한다
prompt_v1 = ChatPromptTemplate.from_messages(
    [
        ("system", "주어진 질문에 한 문단으로 답한다."),
        ("human", "{question}")
    ]
)

# 파서 생성
# StrOutputParser: 복잡한 응답객체에서 응답 텍스트만 추출해주는 도구
parser_v1 = StrOutputParser()

# LCEL을 이용해서 파이프라인 구축
chain_v1 = (prompt_v1 | model | parser_v1).with_config(
    {
        "run_name": "langsmith_trace_practice",
        "tags": ["lecture", "prompt_v1"],
        "metadata": {"prompt_version": "v1"}
    }
)

answer_v1 = chain_v1.invoke({
    "question": "아 진짜 오지게 덥네, 뭐이렇게 더워?"
})

print(answer_v1)




그러게요, 요즘은 가만히 있어도 땀이 날 정도로 너무 덥네요. 물 자주 마시고, 외출할 땐 양산이나 모자 챙기고, 가능하면 낮 시간대엔 시원한 곳에서 쉬세요. 



## trace를 읽는 순서와 판단 기준

trace를 읽을 때는 최종 답변만 보지 않는다. 먼저 최상위 run에서 전체 입력·출력과 오류를 확인한 뒤, run tree를 펼쳐 문제가 시작된 단계를 찾는다.

- **Prompt run**: 입력 변수가 의도한 system·human 메시지로 바뀌었는지 확인한다.
- **Model run**: 모델이 받은 메시지, 생성한 응답, token, cost, latency를 확인한다.
- **Parser run**: `AIMessage`가 최종 문자열로 올바르게 변환됐는지 확인한다.
- **최상위 run**: 전체 체인의 입력·출력과 총 latency, error를 확인한다. 하위 run의 token과 cost가 합산되어 보일 수 있다.

이 예제의 token과 cost는 LLM을 호출한 Model run에서 발생한다. Prompt와 Parser run은 모델을 호출하지 않는다. 화면의 cost는 모델 공급자 API의 예상 비용이며 LangSmith trace 이용 요금과는 다르다. 자세한 기준은 [LangSmith Cost Tracking](https://docs.langchain.com/langsmith/cost-tracking)에서 확인한다.


### run tree에서 단계별 정보 확인하기

새로운 API 호출을 추가하지 않고 앞에서 생성한 `langsmith_trace_practice` trace를 사용한다. LangSmith에서 해당 trace를 열고 다음 내용을 직접 기록한다.

1. 최상위 run의 입력 자료형과 최종 출력 자료형은 무엇인가?
2. Prompt, Model, Parser child run은 어떤 순서로 실행됐는가?
3. Prompt run에서 완성된 system·human 메시지는 무엇인가?
4. token, cost, latency는 어느 child run에서 확인할 수 있는가?
5. Parser run의 입력과 출력은 어떻게 달라졌는가?
6. 최종 답변이 이상하다면 가장 먼저 어느 run의 무엇을 확인하겠는가?

세 단계의 실행 순서, model run의 token·cost, 한 가지 오류 점검 방법을 기록한다.


## 한 trace의 관찰에서 두 조건의 비교로 확장하기

앞에서는 한 trace에서 실행 단계와 문제 위치를 찾았다. 다음에는 같은 질문을 두 조건으로 실행해 출력 품질, token, cost, latency의 차이를 비교한다.

비교할 때는 prompt 외의 질문과 모델을 같게 유지한다. 그래야 관찰된 차이가 prompt 조건에서 비롯됐다고 설명할 수 있다. 같은 기준을 여러 입력과 evaluator에 반복 적용하면 evaluation으로 확장된다.


## 두 조건의 Prompt → Model → Parser 비교하기

같은 질문에 두 prompt 조건을 실행한다.

`brief`는 두 문장 요약을, `example`은 짧은 예시 포함 답변을 요청한다. 각 실행은 Project, 조건 이름, prompt 버전 metadata를 남기며 LCEL의 Prompt → Model → Parser tree를 만든다.

실행 후 다음 순서로 결과를 확인한다.

1. LangSmith에서 `skn33-langsmith-final-practice` Project를 연다.
2. 두 최상위 trace의 latency, token, cost, error, metadata, run tree를 확인한다.
3. 출력 조건을 더 잘 지킨 조건과 근거를 한 문장으로 기록한다.

두 실행의 질문과 모델이 같은지 먼저 확인한 뒤, 출력 조건 준수와 사용량 차이를 근거로 더 적절한 조건을 판단한다.


In [23]:
import langsmith as ls

project_name = "my_test"
question = "LangSmith에서 trace와 run의 관계를 설명해라."

# 조건별 시스템 프롬프트와 비교용 metadata 정의
conditions = {
    "brief": {
        "instruction": "질문에 두 문장 이내로 간결하게 답한다.",
        "metadata":
            {
                "prompt_version": "brief_v2",
                "response_requirement": "two_sentences"
            }
    },
    "example": {
        "instruction": "질문에 세 문장 이내로 답하고 짧은 예시를 하나 포함한다",
        "metadata":
            {
                "prompt_version": "example-v1",
                "response_requirement": "example_in_three_sentence"
            }
    },
}

answers = {}

for condition_name, condition in conditions.items():
    # 같은 입력(question)을 다른 조건(instruction)으로 처리하기
    condition_prompt = ChatPromptTemplate.from_messages([
        ("system", condition["instruction"]),
        ("human", "{question}"),
    ])

    # 처리를 위한 파이프라인 생성
    condition_chain = (condition_prompt | model | StrOutputParser()).with_config({
        "run_name": f"condition_{condition_name}",
        "tags": ["lecture", condition_name],
        "metadata": condition["metadata"]
    })

    # 다른 프로젝트에 추적 내역을 남김
    with ls.tracing_context(project_name=project_name, enabled=True):
        answers[condition_name] = condition_chain.invoke(
            {"question": question}
        )

from pprint import pprint

# for condition_name, answer in answers.items():
#     print(f"### {condition_name}")
#     print(answer)
pprint(answers)

{'brief': 'LangSmith에서 **run**은 LLM 호출, 체인, 도구 실행 등 개별 작업 단위를 나타내며, 각 run은 '
          '입력·출력·지연 시간 같은 정보를 기록합니다. **trace**는 하나의 요청을 처리하는 동안 발생한 루트 run과 '
          '하위(자식) run들을 계층적으로 묶은 전체 실행 흐름입니다.',
 'example': 'LangSmith에서 **run**은 LLM 호출, 체인 실행, 도구 호출 등 하나의 실행 단위이고, '
            '**trace**는 한 요청을 처리하는 동안 발생한 여러 run을 계층 구조로 묶은 전체 실행 기록입니다.  \n'
            '예를 들어 RAG 요청 하나의 trace 안에 `체인 run → 검색기 run → LLM run`이 포함됩니다.'}
